# 02 - Data Preprocessing

**Project:** Multi-Agent Customer Support Intelligence Platform

This notebook prepares the support ticket and FAQ datasets for the next stages:
- Model training
- RAG indexing
- Multi-agent workflow

### Preprocessing steps
1. Load raw datasets
2. Check and remove duplicates
3. Handle missing values
4. Convert date columns
5. Clean ticket and FAQ text
6. Create useful text features
7. Validate categorical values
8. Save cleaned datasets to `data/processed/`

**Note:** We will not generate synthetic data here.

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import re

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

## 1. Define project paths

In [2]:
# Notebook is inside project/notebooks/
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Raw Data Directory:", RAW_DIR)
print("Processed Data Directory:", PROCESSED_DIR)

Project Root: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform
Raw Data Directory: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\raw
Processed Data Directory: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\processed


## 2. Locate input files

The support ticket file is expected in `data/raw/`.

For the FAQ file, the code searches the project `data/` directory recursively. This makes the notebook robust if the FAQ file is stored in `data/knowledge_base/` instead of `data/raw/`.

In [3]:
def find_file(filename_keywords, search_dir):
    """Find the first CSV file whose name contains all given keywords."""
    csv_files = list(search_dir.rglob("*.csv"))

    for file in csv_files:
        name = file.name.lower()
        if all(keyword.lower() in name for keyword in filename_keywords):
            return file

    return None


TICKETS_PATH = find_file(["support_tickets", "10k"], PROJECT_ROOT / "data")
FAQ_PATH = find_file(["faq", "knowledge", "150"], PROJECT_ROOT / "data")

print("Tickets file:", TICKETS_PATH)
print("FAQ file:", FAQ_PATH)

if TICKETS_PATH is None:
    raise FileNotFoundError(
        "Support ticket CSV was not found inside the data folder."
    )

if FAQ_PATH is None:
    raise FileNotFoundError(
        "FAQ CSV was not found inside the data folder."
    )

Tickets file: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\raw\support_tickets_10k.csv
FAQ file: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\knowledge_base\faq_knowledge_base_150.csv


## 3. Load datasets

In [4]:
tickets = pd.read_csv(TICKETS_PATH)
faq = pd.read_csv(FAQ_PATH)

print("Support Tickets:", tickets.shape)
print("FAQ Knowledge Base:", faq.shape)

Support Tickets: (10000, 15)
FAQ Knowledge Base: (150, 4)


## 4. Create a backup copy

In [5]:
tickets_clean = tickets.copy()
faq_clean = faq.copy()

print("Copies created successfully.")

Copies created successfully.


## 5. Check duplicates

In [6]:
print("Ticket duplicate rows:", tickets_clean.duplicated().sum())
print("FAQ duplicate rows:", faq_clean.duplicated().sum())

print(
    "Duplicate ticket IDs:",
    tickets_clean["ticket_id"].duplicated().sum()
)

print(
    "Duplicate FAQ IDs:",
    faq_clean["faq_id"].duplicated().sum()
)

Ticket duplicate rows: 0
FAQ duplicate rows: 0
Duplicate ticket IDs: 0
Duplicate FAQ IDs: 0


## 6. Remove duplicate records

In [7]:
tickets_clean = tickets_clean.drop_duplicates(
    subset="ticket_id"
).copy()

faq_clean = faq_clean.drop_duplicates(
    subset="faq_id"
).copy()

print("Tickets after duplicate removal:", tickets_clean.shape)
print("FAQs after duplicate removal:", faq_clean.shape)

Tickets after duplicate removal: (10000, 15)
FAQs after duplicate removal: (150, 4)


## 7. Check missing values

In [8]:
print("===== TICKETS =====")
display(tickets_clean.isnull().sum())

print("\n===== FAQ =====")
display(faq_clean.isnull().sum())

===== TICKETS =====


ticket_id                      0
created_date                   0
ticket_text                    0
product_name                   0
product_segment                0
ticket_category                0
priority                       0
sentiment                      0
confidence_score               0
escalated                      0
resolution_text                0
resolved_date                  0
resolution_days                0
auto_resolved                  0
customer_satisfaction_score    0
dtype: int64


===== FAQ =====


faq_id      0
question    0
answer      0
category    0
dtype: int64

## 8. Handle missing text values

Text fields should not contain NaN values because they will later be used by:
- ML models
- Embedding models
- RAG retrieval

We replace missing text with an empty string.

In [9]:
ticket_text_columns = [
    "ticket_text",
    "resolution_text"
]

faq_text_columns = [
    "question",
    "answer"
]

for column in ticket_text_columns:
    tickets_clean[column] = tickets_clean[column].fillna("").astype(str)

for column in faq_text_columns:
    faq_clean[column] = faq_clean[column].fillna("").astype(str)

print("Missing text values handled.")

Missing text values handled.


## 9. Clean text

In [10]:
def clean_text(text):
    """Basic text cleaning for support tickets and FAQs."""

    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text


tickets_clean["ticket_text_clean"] = tickets_clean["ticket_text"].apply(clean_text)
tickets_clean["resolution_text_clean"] = tickets_clean["resolution_text"].apply(clean_text)

faq_clean["question_clean"] = faq_clean["question"].apply(clean_text)
faq_clean["answer_clean"] = faq_clean["answer"].apply(clean_text)

display(
    tickets_clean[
        ["ticket_text", "ticket_text_clean"]
    ].head()
)

,ticket_text,ticket_text_clean
0,The lipstick I received in order #ORD5614226 stopped working after just 5 days of use.,the lipstick i received in order #ord5614226 stopped working after just 5 days of use.
1,The shoe rack received in order #ORD8038374 has scratches and dents. Looks like a returned/used item.,the shoe rack received in order #ord8038374 has scratches and dents. looks like a returned/used item.
2,I want to exchange my conditioner (order #ORD6770619) for a different size. Exchange option is missing.,i want to exchange my conditioner (order #ord6770619) for a different size. exchange option is missing.
3,My makeup brush set from order #ORD4823498 appears to be counterfeit/fake. Not the original brand.,my makeup brush set from order #ord4823498 appears to be counterfeit/fake. not the original brand.
4,Product listing for lipstick is completely misleading. The actual product is nothing like the photos.,product listing for lipstick is completely misleading. the actual product is nothing like the photos.


## 10. Convert date columns

Date columns are converted to pandas datetime format so they can be used safely later.

In [11]:
tickets_clean["created_date"] = pd.to_datetime(
    tickets_clean["created_date"],
    errors="coerce"
)

tickets_clean["resolved_date"] = pd.to_datetime(
    tickets_clean["resolved_date"],
    errors="coerce"
)

print("Created date type:", tickets_clean["created_date"].dtype)
print("Resolved date type:", tickets_clean["resolved_date"].dtype)

print(
    "Invalid created dates:",
    tickets_clean["created_date"].isna().sum()
)

print(
    "Invalid resolved dates:",
    tickets_clean["resolved_date"].isna().sum()
)

Created date type: datetime64[ns]
Resolved date type: datetime64[ns]
Invalid created dates: 0
Invalid resolved dates: 0


## 11. Check categorical columns

In [12]:
categorical_columns = [
    "product_name",
    "product_segment",
    "ticket_category",
    "priority",
    "sentiment",
    "escalated",
    "auto_resolved"
]

for column in categorical_columns:
    print(f"\n{column}:")
    print(tickets_clean[column].value_counts(dropna=False))


product_name:
product_name
lipstick              97
soft toy              96
atta (wheat flour)    93
yoga mat              92
bluetooth speaker     92
                      ..
LED TV                62
cricket bat           61
dining table          59
instant noodles       56
self-help book        51
Name: count, Length: 130, dtype: int64

product_segment:
product_segment
Electronics               1573
Beauty & Personal Care    1560
Home & Kitchen            1524
Fashion                   1501
Sports & Fitness          1181
Grocery & Health          1158
Toys & Games               784
Books & Education          719
Name: count, dtype: int64

ticket_category:
ticket_category
Refund & Return             2051
Delivery Issue              1950
Product Issue               1948
Payment Issue               1474
Account & Login             1030
Seller & Product Listing     830
App & Website Issue          717
Name: count, dtype: int64

priority:
priority
High      4620
Medium    3657
Low      

## 12. Handle missing categorical values

If any categorical value is missing, we use `Unknown`.

This avoids deleting tickets unnecessarily.

In [13]:
for column in categorical_columns:
    tickets_clean[column] = tickets_clean[column].fillna("Unknown")

faq_clean["category"] = faq_clean["category"].fillna("Unknown")

print("Categorical missing values handled.")

Categorical missing values handled.


## 13. Check numerical columns

In [14]:
numerical_columns = [
    "confidence_score",
    "resolution_days",
    "customer_satisfaction_score"
]

display(tickets_clean[numerical_columns].describe())

print("\nMissing values:")
display(tickets_clean[numerical_columns].isnull().sum())

,confidence_score,resolution_days,customer_satisfaction_score
count,10000.000000,10000.000000,10000.000000
mean,0.818234,4.815200,3.399900
std,0.098229,2.615709,1.266862
min,0.650000,1.000000,1.000000
25%,0.730000,3.000000,3.000000
50%,0.820000,5.000000,4.000000
75%,0.900000,7.000000,4.000000
max,0.990000,10.000000,5.000000



Missing values:


confidence_score               0
resolution_days                0
customer_satisfaction_score    0
dtype: int64

## 14. Create simple text features

In [15]:
tickets_clean["ticket_word_count"] = (
    tickets_clean["ticket_text_clean"]
    .str.split()
    .str.len()
)

tickets_clean["resolution_word_count"] = (
    tickets_clean["resolution_text_clean"]
    .str.split()
    .str.len()
)

faq_clean["question_word_count"] = (
    faq_clean["question_clean"]
    .str.split()
    .str.len()
)

faq_clean["answer_word_count"] = (
    faq_clean["answer_clean"]
    .str.split()
    .str.len()
)

display(
    tickets_clean[
        [
            "ticket_text_clean",
            "ticket_word_count",
            "resolution_word_count"
        ]
    ].head()
)

,ticket_text_clean,ticket_word_count,resolution_word_count
0,the lipstick i received in order #ord5614226 stopped working after just 5 days of use.,15,9
1,the shoe rack received in order #ord8038374 has scratches and dents. looks like a returned/used item.,16,8
2,i want to exchange my conditioner (order #ord6770619) for a different size. exchange option is missing.,16,9
3,my makeup brush set from order #ord4823498 appears to be counterfeit/fake. not the original brand.,15,8
4,product listing for lipstick is completely misleading. the actual product is nothing like the photos.,15,7


## 16. Check target columns

These columns will be important in the model-training stage.

In [16]:
targets = [
    "ticket_category",
    "priority",
    "sentiment",
    "escalated",
    "auto_resolved"
]

for target in targets:
    print(f"\n{target}")
    print(tickets_clean[target].value_counts())


ticket_category
ticket_category
Refund & Return             2051
Delivery Issue              1950
Product Issue               1948
Payment Issue               1474
Account & Login             1030
Seller & Product Listing     830
App & Website Issue          717
Name: count, dtype: int64

priority
priority
High      4620
Medium    3657
Low       1723
Name: count, dtype: int64

sentiment
sentiment
Negative             3077
Neutral              2732
Slightly Negative    1800
Very Negative        1543
Positive              848
Name: count, dtype: int64

escalated
escalated
No     6622
Yes    3378
Name: count, dtype: int64

auto_resolved
auto_resolved
No     9304
Yes     696
Name: count, dtype: int64


## 17. Final missing-value check

In [17]:
print("===== TICKETS =====")
display(
    tickets_clean.isnull().sum()
    .sort_values(ascending=False)
)

print("\n===== FAQ =====")
display(
    faq_clean.isnull().sum()
    .sort_values(ascending=False)
)

===== TICKETS =====


ticket_id                      0
created_date                   0
ticket_text                    0
product_name                   0
product_segment                0
ticket_category                0
priority                       0
sentiment                      0
confidence_score               0
escalated                      0
resolution_text                0
resolved_date                  0
resolution_days                0
auto_resolved                  0
customer_satisfaction_score    0
ticket_text_clean              0
resolution_text_clean          0
ticket_word_count              0
resolution_word_count          0
dtype: int64


===== FAQ =====


faq_id                 0
question               0
answer                 0
category               0
question_clean         0
answer_clean           0
question_word_count    0
answer_word_count      0
dtype: int64

## 18. Final dataset shapes

In [18]:
print("Processed Tickets Shape:", tickets_clean.shape)
print("Processed FAQ Shape:", faq_clean.shape)

Processed Tickets Shape: (10000, 19)
Processed FAQ Shape: (150, 8)


## 19. Save processed datasets

The cleaned files are saved in:

```text
data/processed/
```

These files will be used by the next stages of the project.

In [19]:
PROCESSED_TICKETS_PATH = PROCESSED_DIR / "support_tickets_processed.csv"
PROCESSED_FAQ_PATH = PROCESSED_DIR / "faq_knowledge_base_processed.csv"

tickets_clean.to_csv(
    PROCESSED_TICKETS_PATH,
    index=False
)

faq_clean.to_csv(
    PROCESSED_FAQ_PATH,
    index=False
)

print("Processed files saved successfully.")

print("\nTickets:", PROCESSED_TICKETS_PATH)
print("FAQ:", PROCESSED_FAQ_PATH)

Processed files saved successfully.

Tickets: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\processed\support_tickets_processed.csv
FAQ: c:\Users\shikh\OneDrive\Desktop\Multi-Agent-Customer-Support-Intelligence-Platform\data\processed\faq_knowledge_base_processed.csv


## 20. Verify saved files

In [20]:
print(
    "Tickets file exists:",
    PROCESSED_TICKETS_PATH.exists()
)

print(
    "FAQ file exists:",
    PROCESSED_FAQ_PATH.exists()
)

if PROCESSED_TICKETS_PATH.exists():
    print(
        "Tickets file size:",
        round(PROCESSED_TICKETS_PATH.stat().st_size / 1024, 2),
        "KB"
    )

if PROCESSED_FAQ_PATH.exists():
    print(
        "FAQ file size:",
        round(PROCESSED_FAQ_PATH.stat().st_size / 1024, 2),
        "KB"
    )

Tickets file exists: True
FAQ file exists: True
Tickets file size: 4396.31 KB
FAQ file size: 39.5 KB
